# Ayurvedic plant ID - model comparison on Kaggle

Before running: Accelerator = GPU, Internet = On, and your `plantid-processed` dataset added as an input.
Run all cells top to bottom. Results are zipped to `results.zip` (download from the Output tab).

In [ ]:
import torch, os
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
os.makedirs('ml', exist_ok=True)

In [ ]:
%%writefile ml/common.py
"""Shared helpers for train.py and evaluate.py (torchvision models, transforms, eval dataset)."""
import os
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import models, transforms

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
ARCHS = {"resnet50": 224, "vgg16": 224, "inception_v3": 299,
         "efficientnet_b0": 224, "mobilenet_v3_large": 224}   # input sizes

def pick_device(pref="auto"):
    """auto = CUDA if present, else CPU. The Apple GPU ('mps') is opt-in (--device mps) because some
    PyTorch/macOS combinations crash natively inside convolutions."""
    if pref != "auto":
        return torch.device(pref)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_model(arch, n_classes, pretrained=True):
    """Returns (model, head_module). Only the head is new; everything else is ImageNet-pretrained."""
    if arch not in ARCHS:
        raise ValueError(f"arch must be one of {list(ARCHS)}")
    w = "DEFAULT" if pretrained else None
    m = models.get_model(arch, weights=w)
    if arch == "inception_v3":
        m.aux_logits = False      # drop the auxiliary classifier so forward() returns plain logits
        m.AuxLogits = None
    if arch in ("resnet50", "inception_v3"):
        m.fc = nn.Linear(m.fc.in_features, n_classes); head = m.fc
    elif arch == "vgg16":
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, n_classes); head = m.classifier[6]
    elif arch == "efficientnet_b0":
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes); head = m.classifier[1]
    else:  # mobilenet_v3_large
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, n_classes); head = m.classifier[3]
    return m, head

def set_backbone_trainable(model, head, flag):
    for p in model.parameters():
        p.requires_grad = flag
    for p in head.parameters():
        p.requires_grad = True

def train_tf(img):
    return transforms.Compose([
        transforms.RandomResizedCrop(img, scale=(0.4, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(0.2),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.05),
        transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.2),
        transforms.RandomGrayscale(0.05),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25),
    ])

def eval_tf(img):
    return transforms.Compose([transforms.Resize((img, img)), transforms.ToTensor(),
                               transforms.Normalize(MEAN, STD)])

def load_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert("RGB")   # honour phone rotation

class EvalDataset(Dataset):
    """root/<class_name>/*.jpg. Folders whose name is not in `classes` get label -1 ('unknown plant')."""
    def __init__(self, root, classes, tf):
        self.tf, self.items, self.unknown_folders = tf, [], []
        idx = {c: i for i, c in enumerate(classes)}
        for d in sorted(os.listdir(root)):
            full = os.path.join(root, d)
            if not os.path.isdir(full):
                continue
            label = idx.get(d, -1)
            if label == -1:
                self.unknown_folders.append(d)
            for f in sorted(os.listdir(full)):
                if os.path.splitext(f)[1].lower() in IMG_EXT and not f.startswith("."):
                    self.items.append((os.path.join(full, f), label))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.tf(load_rgb(path)), label


In [ ]:
%%writefile ml/train.py
"""Fine-tune an ImageNet-pretrained CNN on data/processed.

Smoke test first (1-2 minutes):
    python ml/train.py --arch efficientnet_b0 --epochs 1 --epoch-size 200
Real run:
    python ml/train.py --arch efficientnet_b0 --epochs 12
Architectures: resnet50, vgg16, inception_v3, efficientnet_b0, mobilenet_v3_large
(vgg16 is very heavy on a laptop - use Kaggle/Colab for it.)

Phase 1 trains only the new classifier head (--freeze-epochs), phase 2 fine-tunes everything.
Classes are balanced with a weighted sampler. The best epoch by validation macro-F1 is kept.
"""
import argparse, csv, json, os, time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets

import common

def run_epoch(model, loader, device, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, n, ys, ps, top3 = 0.0, 0, [], [], 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y); n += len(y)
            k = min(3, out.shape[1])
            top3 += (out.topk(k, dim=1).indices == y[:, None]).any(dim=1).sum().item()
            ys += y.tolist(); ps += out.argmax(1).tolist()
    return dict(loss=total_loss / max(n, 1), acc=accuracy_score(ys, ps),
                f1=f1_score(ys, ps, average="macro", zero_division=0), top3=top3 / max(n, 1))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/processed"); ap.add_argument("--out", default="models")
    ap.add_argument("--arch", default="efficientnet_b0", choices=list(common.ARCHS))
    ap.add_argument("--epochs", type=int, default=12); ap.add_argument("--freeze-epochs", type=int, default=2)
    ap.add_argument("--bs", type=int, default=32); ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--lr-head", type=float, default=1e-3)
    ap.add_argument("--epoch-size", type=int, default=3000, help="images drawn per epoch (speeds up CPU runs)")
    ap.add_argument("--patience", type=int, default=4); ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda", "mps"])
    a = ap.parse_args()
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    device = common.pick_device(a.device); img = common.ARCHS[a.arch]
    print(f"device={device}  arch={a.arch}  input={img}px")

    tr = datasets.ImageFolder(os.path.join(a.data, "train"), common.train_tf(img))
    va = datasets.ImageFolder(os.path.join(a.data, "val"), common.eval_tf(img))
    te = datasets.ImageFolder(os.path.join(a.data, "test"), common.eval_tf(img))
    assert tr.classes == va.classes == te.classes, "train/val/test class folders differ"
    classes = tr.classes
    json.dump(classes, open(os.path.join(a.out, "classes.json"), "w"), indent=1)

    counts = np.bincount(tr.targets, minlength=len(classes))
    weights = 1.0 / counts[np.array(tr.targets)]
    sampler = WeightedRandomSampler(weights, num_samples=min(len(tr), a.epoch_size), replacement=True)
    kw = dict(num_workers=a.workers, persistent_workers=a.workers > 0)
    tr_dl = DataLoader(tr, a.bs, sampler=sampler, **kw)
    va_dl = DataLoader(va, a.bs, shuffle=False, **kw)
    te_dl = DataLoader(te, a.bs, shuffle=False, **kw)

    model, head = common.build_model(a.arch, len(classes), pretrained=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    ckpt_path = os.path.join(a.out, f"{a.arch}_best.pt")
    best_f1, bad, hist, t0 = -1.0, 0, [], time.time()
    opt = sched = None
    for epoch in range(a.epochs):
        if epoch == 0 and a.freeze_epochs > 0:
            common.set_backbone_trainable(model, head, False)
            opt = torch.optim.AdamW(head.parameters(), lr=a.lr_head, weight_decay=1e-4); phase = "head"
        if epoch == a.freeze_epochs:
            common.set_backbone_trainable(model, head, True)
            opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, a.epochs - a.freeze_epochs))
            phase = "full"
        t = run_epoch(model, tr_dl, device, criterion, opt)
        v = run_epoch(model, va_dl, device, criterion)
        if sched is not None and phase == "full":
            sched.step()
        hist.append(dict(epoch=epoch + 1, phase=phase, train_loss=t["loss"], train_acc=t["acc"],
                         val_loss=v["loss"], val_acc=v["acc"], val_f1=v["f1"], val_top3=v["top3"]))
        print(f"epoch {epoch+1:>2}/{a.epochs} [{phase}] train loss {t['loss']:.3f} acc {t['acc']:.3f} | "
              f"val loss {v['loss']:.3f} acc {v['acc']:.3f} macroF1 {v['f1']:.3f} top3 {v['top3']:.3f} "
              f"| {time.time()-t0:.0f}s", flush=True)
        if v["f1"] > best_f1:
            best_f1, bad = v["f1"], 0
            torch.save(dict(state_dict=model.state_dict(), classes=classes, arch=a.arch, img=img), ckpt_path)
        elif phase == "full":
            bad += 1
            if bad >= a.patience:
                print("early stop"); break

    with open(os.path.join(a.out, f"{a.arch}_history.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(hist[0])); w.writeheader(); w.writerows(hist)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
    test = run_epoch(model, te_dl, device, criterion)
    res = dict(arch=a.arch, classes=len(classes), best_val_macro_f1=best_f1, test_top1=test["acc"],
               test_top3=test["top3"], test_macro_f1=test["f1"], train_seconds=round(time.time() - t0))
    json.dump(res, open(os.path.join(a.out, f"{a.arch}_metrics.json"), "w"), indent=1)
    print("TEST (held-out MED117 blocks / Mendeley splits - optimistic, see report):", res)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/evaluate.py
"""Evaluate a trained checkpoint on any folder laid out as <root>/<class_id>/*.jpg.

    python ml/evaluate.py --ckpt models/efficientnet_b0_best.pt --root data/processed/test --name test
    python ml/evaluate.py --ckpt models/efficientnet_b0_best.pt --root data/field_test    --name field

Writes reports/<name>_<arch>/: report.txt (top-1, top-3, per-class precision/recall/F1),
confusion_matrix.png, predictions.csv (every image, with top-3), thresholds.csv.
Folders whose name is NOT a trained class are treated as 'unknown plants': they are left out of
accuracy, and thresholds.csv shows how often the model confidently (wrongly) names them.
That table is how you choose the confidence cut-off for the app's 'Not sure' state.
"""
import argparse, csv, os
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

import common

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True); ap.add_argument("--root", required=True)
    ap.add_argument("--name", default="eval"); ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda", "mps"])
    a = ap.parse_args()
    device = common.pick_device(a.device)
    ck = torch.load(a.ckpt, map_location="cpu")
    classes, arch, img = ck["classes"], ck["arch"], ck["img"]
    model, _ = common.build_model(arch, len(classes), pretrained=False)
    model.load_state_dict(ck["state_dict"]); model.to(device).eval()

    ds = common.EvalDataset(a.root, classes, common.eval_tf(img))
    if ds.unknown_folders:
        print("Treating as UNKNOWN plants (not trained classes):", ds.unknown_folders)
    dl = DataLoader(ds, a.bs, shuffle=False, num_workers=a.workers)
    probs, labels = [], []
    with torch.no_grad():
        for x, y in dl:
            probs.append(torch.softmax(model(x.to(device)), 1).cpu().numpy()); labels += y.tolist()
    P, y = np.concatenate(probs), np.array(labels)
    paths = [p for p, _ in ds.items]
    top = np.argsort(-P, axis=1)[:, :3]
    known = y >= 0
    out_dir = os.path.join("reports", f"{a.name}_{arch}"); os.makedirs(out_dir, exist_ok=True)

    lines = [f"checkpoint={a.ckpt}  root={a.root}  images={len(y)}  known={known.sum()}  unknown={(~known).sum()}"]
    if known.any():
        top1 = (top[known, 0] == y[known]).mean(); top3 = (top[known] == y[known][:, None]).any(1).mean()
        present = sorted(set(y[known].tolist()))
        lines += [f"top-1 accuracy: {top1:.4f}", f"top-3 accuracy: {top3:.4f}", "",
                  classification_report(y[known], top[known, 0], labels=present,
                                        target_names=[classes[i] for i in present], zero_division=0)]
        cm = confusion_matrix(y[known], top[known, 0], labels=present).astype(float)
        cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        fig, ax = plt.subplots(figsize=(max(6, len(present) * 0.45), max(5, len(present) * 0.4)))
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(present))); ax.set_yticks(range(len(present)))
        ax.set_xticklabels([classes[i] for i in present], rotation=90, fontsize=7)
        ax.set_yticklabels([classes[i] for i in present], fontsize=7)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{a.name}: row-normalised confusion matrix")
        fig.tight_layout(); fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150); plt.close(fig)
    with open(os.path.join(out_dir, "report.txt"), "w") as f:
        f.write("\n".join(lines))
    print("\n".join(lines))

    with open(os.path.join(out_dir, "predictions.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["path", "true", "pred1", "conf1", "pred2", "conf2", "pred3", "conf3"])
        for i, p in enumerate(paths):
            row = [p, classes[y[i]] if y[i] >= 0 else "UNKNOWN"]
            for j in top[i]:
                row += [classes[j], f"{P[i, j]:.3f}"]
            w.writerow(row)

    maxp = P.max(1)
    with open(os.path.join(out_dir, "thresholds.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["threshold", "known_coverage", "known_acc_when_answered", "unknown_wrongly_accepted"])
        for t in [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            acc_mask = known & (maxp >= t)
            cov = acc_mask.sum() / max(known.sum(), 1)
            acc = (top[acc_mask, 0] == y[acc_mask]).mean() if acc_mask.any() else float("nan")
            unk = ((~known) & (maxp >= t)).sum() / max((~known).sum(), 1) if (~known).any() else float("nan")
            w.writerow([t, f"{cov:.3f}", f"{acc:.3f}", f"{unk:.3f}"])
    print(f"\nreports written to {out_dir}/")

if __name__ == "__main__":
    main()


In [ ]:
import glob, os
# find the folder that contains train/ val/ test/ inside the attached dataset
roots = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train', recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), 'val')) and os.path.isdir(os.path.join(os.path.dirname(p), 'test'))]
assert roots, 'Could not find train/val/test - is the dataset attached as an input?'
DATA = roots[0]
print('DATA =', DATA)
for s in ('train','val','test'):
    print(s, sum(len(f) for _,_,f in os.walk(os.path.join(DATA, s))), 'images')

## Train each architecture
Each run keeps the best epoch by validation macro-F1. Add `'vgg16'` to the list if you have time (slow, very large).

In [ ]:
ARCHS = ['efficientnet_b0', 'mobilenet_v3_large', 'resnet50', 'inception_v3']
EPOCHS = 15
for arch in ARCHS:
    print('='*30, arch, '='*30)
    !python ml/train.py --data {DATA} --arch {arch} --epochs {EPOCHS} --workers 2 --out models

## Evaluate on the held-out test split
Remember: MED117 test accuracy is optimistic (only 2-3 leaves per species). The field test set is the honest number.

In [ ]:
for arch in ARCHS:
    print('='*30, arch, '='*30)
    !python ml/evaluate.py --ckpt models/{arch}_best.pt --root {DATA}/test --name test --workers 2

In [ ]:
import json, pandas as pd
rows = [json.load(open(f'models/{a}_metrics.json')) for a in ARCHS if os.path.exists(f'models/{a}_metrics.json')]
df = pd.DataFrame(rows).round(4)
df.to_csv('models/comparison.csv', index=False)
df

In [ ]:
!zip -rq results.zip models reports -x '*.DS_Store'
!ls -lh results.zip